In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=PYfveeI8Qdk50KC9gj387SP6gp36qb&access_type=offline&code_challenge=WgsbJpHRZr13wFF87tAE4kJf58jAeMm8Vw4RfBJs4RA&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


path_to_release_folder="gs://open-targets-data-releases/25.03/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

Loading BokehJS ...

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/13 16:26:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/13 16:26:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/13 16:26:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


# Estimated MAF and effect size distribution

In [39]:
df=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/rescaled-betas.parquet").cache()

25/05/13 12:42:50 WARN CacheManager: Asked to cache already cached data.


In [44]:
df.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- zScore: double (nullable = true)
 |-- pValueMantissa: float (nullable = true)
 |-- pValueExponent: integer (nullable = true)
 |-- standardError: double (nullable = true)
 |-- finemappingMethod: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- credibleSetSize: integer (nullable = true)
 |-- posteriorProbability: double (nullable = true)
 |-- nSamples: integer (nullable = true)
 |-- nControls: integer (nullable = true)
 |-- nCases: integer (nullable = true)
 |-- majorPopulation: struct (nullable = true)
 |    |-- ldPopulation: string (nullable = true)
 |    |-- relativeSampleSize: double (nullable = true)
 |-- allelefrequencies: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- populationName: string (nullable = true)
 |    |    |-- alleleFrequency: double (nulla

In [40]:
repl_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_gwas_replicated_CSs.parquet")

In [41]:
df_gwas_repl=df.join(repl_cs,on="studyLocusId",how="inner").filter(f.col("studyType")=="gwas").cache()
df_gwas_repl.count()

25/05/13 12:42:51 WARN CacheManager: Asked to cache already cached data.


254836

In [46]:
df_gwas_repl.filter((f.col("rescaledStatistics.estimatedBeta")<=-0.5) | (f.col("rescaledStatistics.estimatedBeta")>=0.5)).count()

11542

In [45]:
df_gwas_repl.filter((f.col("rescaledStatistics.estimatedBeta")<0.5) & (f.col("rescaledStatistics.estimatedBeta")>-0.5)).count()

242927

In [42]:
df_gwas_repl.filter(f.col("majorPopulationMAF")<0.01).count()

15763

In [43]:
df_gwas_repl.filter(f.col("majorPopulationMAF")>=0.01).count()

238706

In [47]:
train_set_full=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/2506_traing_set.json")

In [38]:
train_set_full.filter(f.col("goldStandardSet")=="positive").select("diseaseIds","geneId").distinct().count()

1408

In [48]:
train_set_full.printSchema()

root
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- geneId: string (nullable = true)
 |-- goldStandardSet: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- variantId: string (nullable = true)



In [49]:
train_set_full=train_set_full.filter(f.col("goldStandardSet")=="positive")

In [50]:
train_set_full.count()

7448

In [51]:
train_set_full=train_set_full.join(df,on="studyLocusId",how="inner")
train_set_full.count()

7358

In [52]:
train_set_full.filter(f.col("majorPopulationMAF")<0.01).count()

654

In [53]:
train_set_full.filter(f.col("majorPopulationMAF")>=0.01).count()

6702

In [54]:
train_set_full.filter((f.col("rescaledStatistics.estimatedBeta")<=-0.5) | (f.col("rescaledStatistics.estimatedBeta")>=0.5)).count()

591

In [55]:
train_set_full.filter((f.col("rescaledStatistics.estimatedBeta")<0.5) & (f.col("rescaledStatistics.estimatedBeta")>-0.5)).count()

6765

In [13]:
train_set=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/2506_traing_set_dedupl.json")

In [14]:
train_set=train_set.filter(f.col("goldStandardSet")=="positive").join(df,on="studyLocusId",how="inner")
train_set.count()

4558

In [17]:
train_set.filter(f.col("majorPopulationMAF")<0.01).count()

457

In [18]:
train_set.filter(f.col("majorPopulationMAF")>0.01).count()

4099

# Estimate the nearest gene

In [2]:
train_set=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/2506_traing_set_dedupl.json")

In [3]:
train_set.count()

51072

In [4]:
fm_0=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/2506_traing_set_full_fm.parquet")

In [5]:
fm_0.count()

82266

In [6]:
fm_0=fm_0.join(train_set.select("studyLocusId").distinct(), on="studyLocusId", how="inner").cache()
fm_0.count()

25/05/13 12:57:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


51072

In [7]:
from pyspark.sql.functions import concat, lit, array_join

fm_1=fm_0.filter(f.col("GSP")==1).withColumn(
    "GSP_efo",
    concat(f.col("geneId"), lit("_"), array_join(f.col("diseaseIds"), ","))
).select("studyLocusId","GSP_efo").cache()
fm_1.show(2,truncate=False)

+--------------------------------+---------------------------------------+
|studyLocusId                    |GSP_efo                                |
+--------------------------------+---------------------------------------+
|028243d6514e41ef0c9e571a10dcadee|ENSG00000021826_EFO_0009767            |
|0b022fe83fc2b9304f9835d401aa777b|ENSG00000101670_EFO_0004612,EFO_0010351|
+--------------------------------+---------------------------------------+
only showing top 2 rows



In [8]:
fm_1.count()

4597

In [9]:
fm_0.count()

51072

In [10]:
fm_0=fm_0.join(fm_1, on="studyLocusId", how="inner").cache()
fm_0.count()

51072

## G-D-V

In [22]:
fm_0.filter(f.col("GSP")==1).count()

4597

In [19]:
tmp=fm_0.filter(f.col("GSP")==1)
n1=tmp.count()
n2=(tmp
.filter(
    (f.col("distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("distanceTssMeanNeighbourhood")==1)).count()
)


gsp_efo_to_exclude=tmp.filter(
    (f.col("distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("distanceTssMeanNeighbourhood")==1)).select("studyLocusId")


n2/n1

0.6090928866652164

In [23]:
gsp_efo_to_exclude.count()

2800

In [28]:
tmp=(fm_0
.join(gsp_efo_to_exclude, on="studyLocusId", how="anti")
.filter(f.col("GSP")==0))

n1=tmp.select("studyLocusId").distinct().count()

n2=(tmp
.filter(
    (f.col("distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("distanceTssMeanNeighbourhood")==1)
    ).select("studyLocusId").distinct().count()
)
n2/n1

0.5361305361305362

# G-D

In [59]:
tmp=fm_0.filter(f.col("GSP")==1)

tmp=(tmp.groupBy("diseaseIds","geneId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

gsp_efo_to_exclude=tmp2.select("diseaseIds","geneId").distinct()
studyLocusId_to_exclude=fm_0.filter(f.col("GSP")==1).join(gsp_efo_to_exclude, on=["diseaseIds","geneId"], how="inner").select("studyLocusId").distinct()

n2/n1

0.6285511363636364

In [60]:
fm_0.filter(f.col("GSP")==1).join(studyLocusId_to_exclude, on="studyLocusId", how="inner").select("diseaseIds","geneId").distinct().count()

885

In [62]:
fm_0.select("GSP_efo").distinct().count()

1408

In [63]:
tmp=(fm_0
.join(studyLocusId_to_exclude, on="studyLocusId", how="anti"))

tmp_g=tmp.filter(f.col("GSP")==1).select("GSP_efo").distinct()
n1=tmp_g.count()

tmp2=(tmp.groupBy("GSP_efo")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)



tmp3=(tmp2
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp3.count()

n2/n1


0.5927342256214149

In [68]:
n1

523

# G-D on big matrix

In [72]:
fm=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/not_filltered_FM_with_GSP.parquet")

In [73]:
from pyspark.sql.functions import concat, lit, array_join

fm_1=fm.filter(f.col("GSP")==1).withColumn(
    "GSP_efo",
    concat(f.col("geneId"), lit("_"), array_join(f.col("diseaseIds"), ","))
).select("studyLocusId","GSP_efo").cache()
fm_1.show(2,truncate=False)

+--------------------------------+---------------------------------------+
|studyLocusId                    |GSP_efo                                |
+--------------------------------+---------------------------------------+
|062843a9f56cafe53ea708f396a878e7|ENSG00000169174_EFO_0004611,EFO_0004639|
|0f4223ef92ec2b5366c3f3a20ffde5e3|ENSG00000087237_EFO_0004612            |
+--------------------------------+---------------------------------------+
only showing top 2 rows



In [76]:
list_of_gsps=fm_0.filter(f.col("GSP")==1).select("GSP_efo").distinct()

In [78]:
fm2=fm_1.join(list_of_gsps, on="GSP_efo", how="inner").select("studyLocusId").distinct()
fm2.count()

19183

In [80]:
fm3=fm.join(fm2, on="studyLocusId", how="inner").filter(f.col("isProteinCoding")==1).cache()
fm3.count()

310634

In [81]:
tmp=fm3.filter(f.col("GSP")==1)

tmp=(tmp.groupBy("diseaseIds","geneId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

gsp_efo_to_exclude=tmp2.select("diseaseIds","geneId").distinct()
studyLocusId_to_exclude=fm3.filter(f.col("GSP")==1).join(gsp_efo_to_exclude, on=["diseaseIds","geneId"], how="inner").select("studyLocusId").distinct()

n2/n1

0.6646626586506346

# Nearest of L2G score

In [2]:
# combinig it with l2g predictions
l2g=session.spark.read.parquet(path_to_release_folder+"output/l2g_prediction")

In [3]:
qsi=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_studies_with_oncology")
qsi_measurements=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/qualified_studies_measurements")

In [4]:
qsi=qsi.union(qsi_measurements).distinct().cache()
qsi.count()

67751

In [5]:
repl_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_gwas_replicated_CSs.parquet")

In [6]:
l2g=l2g.join(sl.df.select("studyLocusId","studyId","variantId"), on="studyLocusId", how="inner").drop("shapBaseValue")
l2g=l2g.join(si.df.select("studyId","diseaseIds"), on="studyId", how="inner").cache()
l2g.count()


992067

In [7]:
l2g=l2g.join(qsi, on="studyId", how="inner")
l2g=l2g.join(repl_cs, on="studyLocusId", how="inner").cache()
l2g.count()

301998

In [8]:
l2g.printSchema()

root
 |-- studyLocusId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- score: double (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- value: float (nullable = true)
 |    |    |-- shapValue: float (nullable = true)
 |-- variantId: string (nullable = true)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [11]:
l2g.select("studyLocusId").distinct().count()

176249

In [9]:
from pyspark.sql import Window
import pyspark.sql.functions as f

# Define a Window specification to partition by studyLocusId and order by score in descending order
window_spec = Window.partitionBy("studyLocusId").orderBy(f.col("score").desc())

# Add a row number column to identify the top geneId per studyLocusId
l2g_with_row_number = l2g.withColumn("row_number", f.row_number().over(window_spec))

# Filter to keep only the top row (row_number == 1) for each studyLocusId
l2g_top_gene = l2g_with_row_number.filter(f.col("row_number") == 1).drop("row_number").cache()

# Show the result
l2g_top_gene.count()

176249

In [12]:
l2g_top_gene.show(1)

+--------------------+------------+---------------+------------------+--------------------+---------------+-------------+
|        studyLocusId|     studyId|         geneId|             score|            features|      variantId|   diseaseIds|
+--------------------+------------+---------------+------------------+--------------------+---------------+-------------+
|0039553eb290818c8...|GCST90435412|ENSG00000085276|0.8148451518774646|[{eQtlColocClppMa...|3_169520227_T_C|[EFO_0004339]|
+--------------------+------------+---------------+------------------+--------------------+---------------+-------------+
only showing top 1 row



In [13]:
l2g_0_5=l2g.filter(f.col("score")>=0.5).cache()
l2g_0_5.count()

142298

In [14]:
fm=session.spark.read.parquet(path_to_release_folder+"intermediate/l2g_feature_matrix/")

# Top l2g gene per CS

In [15]:
l2g_top_gene=l2g_top_gene.join(fm,on=["studyLocusId","geneId"],how="inner").cache()
l2g_top_gene.count()

25/05/13 16:39:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


176249

In [17]:
tmp=l2g_top_gene

tmp=(tmp.groupBy("diseaseIds","geneId","variantId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

n2/n1

0.9483395916451537

In [18]:
tmp=l2g_top_gene

tmp=(tmp.groupBy("diseaseIds","geneId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

n2/n1

0.9508750383788762

# L2G>=0.5 

In [20]:
l2g_0_5=l2g_0_5.join(fm,on=["studyLocusId","geneId"],how="inner").cache()
l2g_0_5.count()

142298

In [ ]:
tmp=l2g_0_5

tmp=(tmp.groupBy("diseaseIds","geneId","variantId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

n2/n1

0.9360317836379872

In [22]:
tmp=l2g_0_5

tmp=(tmp.groupBy("diseaseIds","geneId")
.agg(
    f.max("distanceSentinelFootprintNeighbourhood").alias("max_distanceSentinelFootprintNeighbourhood"),
    f.max("distanceSentinelTssNeighbourhood").alias("max_distanceSentinelTssNeighbourhood"),
    f.max("distanceFootprintMeanNeighbourhood").alias("max_distanceFootprintMeanNeighbourhood"),
    f.max("distanceTssMeanNeighbourhood").alias("max_distanceTssMeanNeighbourhood")
    )
)

n1=tmp.count()

tmp2=(tmp
.filter(
    (f.col("max_distanceSentinelFootprintNeighbourhood")==1) 
    | 
    (f.col("max_distanceSentinelTssNeighbourhood")==1)
    |
    (f.col("max_distanceFootprintMeanNeighbourhood")==1)
    |
    (f.col("max_distanceTssMeanNeighbourhood")==1))
)

n2=tmp2.count()

n2/n1

0.9399073425631633